# 5. Classificare e segmentare un PDF multi-documento

Classifica i documenti nel PDF e ne identifica gli intervalli di pagina.

## 1. Import e configurazione

Carica librerie, variabili di ambiente, schema e documento.

In [1]:
import json
import os
from pathlib import Path
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import ContentAnalyzer
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

In [ ]:
load_dotenv(r"..\.env", override=True)
DOCUMENTO = Path(r"..\file\esercizio5.pdf")
document_bytes = DOCUMENTO.read_bytes()

# Legge la definizione JSON, non il risultato di un'analisi.
definizione = json.loads(
    Path(r"..\modelli\classificatore_documenti.json").read_text(encoding="utf-8")
)

## 2. Client ed esecuzione

Crea il classificatore temporaneo, analizza il PDF e rimuove il classificatore.

In [ ]:
# Crea client e credenziale; with li chiude anche in caso di errore.
with DefaultAzureCredential() as credential, ContentUnderstandingClient(
    endpoint=os.environ["FOUNDRY_SERVICES_ENDPOINT"], credential=credential,
) as client:
    # Crea l'analyzer dal JSON; allow_replace aggiorna l'ID del laboratorio.
    client.begin_create_analyzer(
        analyzer_id="classificatore_documenti", resource=ContentAnalyzer(definizione), allow_replace=True,
    ).result()
    try:
        # Analizza i byte del PDF con l'analyzer indicato e attende il risultato.
        result = client.begin_analyze_binary(
            analyzer_id="classificatore_documenti", binary_input=document_bytes,
        ).result()
    finally:
        # Elimina l'analyzer temporaneo identificato da analyzer_id.
        client.delete_analyzer(analyzer_id="classificatore_documenti")

## 3. Risultato

Mostra categoria, pagine e confidence di ogni segmento.

In [4]:
# as_dict espone categoria, pagine e confidence restituite dal servizio.
segmenti = [
    {
        "categoria": segment["category"],
        "pagina_iniziale": segment["startPageNumber"],
        "pagina_finale": segment["endPageNumber"],
        "confidence": segment["confidence"],
    }
    for content in result.as_dict()["contents"]
    for segment in content["segments"]
]
print(json.dumps(segmenti, ensure_ascii=False, indent=2))

[
  {
    "categoria": "hotel_receipt",
    "pagina_iniziale": 1,
    "pagina_finale": 2,
    "confidence": 0.98
  },
  {
    "categoria": "restaurant_receipt",
    "pagina_iniziale": 3,
    "pagina_finale": 3,
    "confidence": 0.99
  },
  {
    "categoria": "other",
    "pagina_iniziale": 4,
    "pagina_finale": 4,
    "confidence": 0.99
  }
]
